In [1]:
def naive_with_counts(p, t):
    occurrences = []
    num_alignments = 0
    num_character_comparisons = 0
    
    for i in range(len(t) - len(p) + 1):
        num_alignments += 1
        mismatched = False
        for j in range(len(p)):
            num_character_comparisons += 1
            if t[i+j] != p[j]:
                mismatched = True
                break
        if not mismatched:
            occurrences.append(i)
            
    return occurrences, num_alignments, num_character_comparisons

In [2]:
def boyer_moore_with_counts(p, p_bm, t):
    """ Do Boyer-Moore matching with alignment and comparison trackers.
        p=pattern, t=text, p_bm=BoyerMoore object for p """
    i = 0
    occurrences = []
    num_alignments = 0
    num_character_comparisons = 0
    
    while i < len(t) - len(p) + 1:
        num_alignments += 1
        shift = 1
        mismatched = False
        for j in range(len(p)-1, -1, -1):
            num_character_comparisons += 1
            if p[j] != t[i+j]:
                skip_bc = p_bm.bad_character_rule(j, t[i+j])
                skip_gs = p_bm.good_suffix_rule(j)
                shift = max(shift, skip_bc, skip_gs)
                mismatched = True
                break
        if not mismatched:
            occurrences.append(i)
            skip_gs = p_bm.match_skip()
            shift = max(shift, skip_gs)
        i += shift
        
    return occurrences, num_alignments, num_character_comparisons

In [3]:
def read_fasta(filename):
    """Reads a FASTA file and returns the sequence as a single string."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            # Skip the header line
            if line.startswith('>'):
                continue
            # Remove whitespace/newlines and add to list
            sequence.append(line.strip())
    return ''.join(sequence)

In [4]:
# Assuming you wrote a helper to read the FASTA file string into `t`
t = read_fasta('chr1.GRCh38.excerpt.fasta')
p = 'GGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGAGG' 

# For Boyer-Moore:
from bm_preproc import BoyerMoore
p_bm = BoyerMoore(p, alphabet='ACGT')

occ, alignments, comparisons = boyer_moore_with_counts(p, p_bm, t)
# Print the results
print(f"Occurrences: {occ}")   
print(f"Number of alignments: {alignments}")
print(f"Number of character comparisons: {comparisons}")
# for naive:
occ_naive, alignments_naive, comparisons_naive = naive_with_counts(p, t)
print(f"Occurrences (naive): {occ_naive}")
print(f"Number of alignments (naive): {alignments_naive}")
print(f"Number of character comparisons (naive): {comparisons_naive}")

Occurrences: [56922]
Number of alignments: 127974
Number of character comparisons: 165191
Occurrences (naive): [56922]
Number of alignments (naive): 799954
Number of character comparisons (naive): 984143


In [5]:
import bisect

# Assuming the Index class from kmer_index.py is loaded or defined as follows
class Index(object):
    def __init__(self, t, k):
        self.k = k
        self.index = []
        for i in range(len(t) - k + 1):
            self.index.append((t[i:i+k], i))
        self.index.sort()
    
    def query(self, p):
        kmer = p[:self.k]
        i = bisect.bisect_left(self.index, (kmer, -1))
        hits = []
        while i < len(self.index):
            if self.index[i][0] != kmer:
                break
            hits.append(self.index[i][1])
            i += 1
        return hits

def query_index_approximate(p, t, index, max_mismatches=2):
    """Finds all occurrences of P in T with up to max_mismatches using the Index."""
    k = index.k  # This is 8
    partition_len = k
    occurrences = set()  # Use a set to prevent counting the same starting index multiple times
    
    # P has length 24, k=8 -> exactly 3 partitions
    for part_num in range(3):
        start_p = part_num * partition_len
        end_p = start_p + partition_len
        partition = p[start_p:end_p]
        
        # Query the index with the 8-mer partition
        hits = index.query(partition)
        
        for hit in hits:
            # Calculate where the full pattern P would start in T
            start_t = hit - start_p
            
            # Ensure the full pattern fits within the text boundaries
            if start_t < 0 or start_t + len(p) > len(t):
                continue
                
            # Verify the alignment and count mismatches
            mismatches = 0
            for j in range(len(p)):
                if t[start_t + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break
            
            if mismatches <= max_mismatches:
                occurrences.add(start_t)
                
    return sorted(list(occurrences))

In [6]:
# Load the text from the FASTA file
t = read_fasta('chr1.GRCh38.excerpt.fasta')

# Build the 8-mer index
idx = Index(t, 8)

# Define the pattern
p = 'GGCGCGGTGGCTCACGCCTGTAAT'

# Run the approximate matching algorithm
matches = query_index_approximate(p, t, idx, max_mismatches=2)

print(f"Occurrences found at indices: {matches}")
print(f"Total number of times the pattern occurs: {len(matches)}")

Occurrences found at indices: [56922, 84641, 147558, 160162, 160729, 191452, 262042, 273669, 364263, 421221, 429299, 465647, 551134, 635931, 657496, 681737, 717706, 724927, 747359]
Total number of times the pattern occurs: 19


In [7]:
def count_total_index_hits(p, index):
    k = index.k  # 8
    partition_len = k
    total_hits = 0
    
    # Loop through all 3 partitions of the 24-mer pattern
    for part_num in range(3):
        start_p = part_num * partition_len
        end_p = start_p + partition_len
        partition = p[start_p:end_p]
        
        # Query the index and add the number of hits found
        hits = index.query(partition)
        total_hits += len(hits)
        
    return total_hits

In [ ]:
# Complete demonstration script to verify both the occurrences and the total index hits

import bisect

class Index(object):
    def __init__(self, t, k):
        self.k = k
        self.index = []
        for i in range(len(t) - k + 1):
            self.index.append((t[i:i+k], i))
        self.index.sort()
    
    def query(self, p):
        kmer = p[:self.k]
        i = bisect.bisect_left(self.index, (kmer, -1))
        hits = []
        while i < len(self.index):
            if self.index[i][0] != kmer:
                break
            hits.append(self.index[i][1])
            i += 1
        return hits

def read_fasta(filename):
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('>'):
                continue
            sequence.append(line.strip())
    return ''.join(sequence)

def query_index_with_hit_count(p, t, index, max_mismatches=2):
    k = index.k
    partition_len = k
    occurrences = set()
    total_hits = 0
    
    # 3 partitions for a 24-mer pattern using an 8-mer index
    for part_num in range(3):
        start_p = part_num * partition_len
        end_p = start_p + partition_len
        partition = p[start_p:end_p]
        
        # Query index
        hits = index.query(partition)
        total_hits += len(hits)  # Count every single raw hit
        
        for hit in hits:
            start_t = hit - start_p
            if start_t < 0 or start_t + len(p) > len(t):
                continue
                
            # Verification step
            mismatches = 0
            for j in range(len(p)):
                if t[start_t + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break
            if mismatches <= max_mismatches:
                occurrences.add(start_t)
                
    return sorted(list(occurrences)), total_hits




In [ ]:
# --- Execution ---
t = read_fasta('chr1.GRCh38.excerpt.fasta')
idx = Index(t, 8)
p = 'GGCGCGGTGGCTCACGCCTGTAAT'
matches, hits_count = query_index_with_hit_count(p, t, idx)

print(f"Total Matches Found: {len(matches)}")
print(f"Total Index Hits Traversed: {hits_count}")

Total Matches Found: 19
Total Index Hits Traversed: 90


In [10]:
def query_subseq_index(p, t, subseq_index, max_mismatches=2):
    """
    Finds approximate occurrences and counts total raw index hits 
    using a SubseqIndex with k=8 and ival=3.
    """
    occurrences = set()
    total_hits = 0
    
    # Since ival=3, we have 3 subsequence partitions starting at offsets 0, 1, and 2
    for start_offset in range(3):
        # The query method internally slices p[start_offset : start_offset + span : ival]
        hits = subseq_index.query(p[start_offset:])
        total_hits += len(hits)
        
        for hit in hits:
            # The hit index corresponds to where the subsequence starts in text T.
            # To get the alignment index of the entire pattern P, we subtract the start_offset.
            start_t = hit - start_offset
            
            if start_t < 0 or start_t + len(p) > len(t):
                continue
                
            # Verify the full alignment against the mismatch limit
            mismatches = 0
            for j in range(len(p)):
                if t[start_t + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break
                        
            if mismatches <= max_mismatches:
                occurrences.add(start_t)
                
    return sorted(list(occurrences)), total_hits

In [11]:
# Complete runnable verification script using SubseqIndex

import bisect

class SubseqIndex(object):
    """ Holds a subsequence index for a text T """
    
    def __init__(self, t, k, ival):
        """ Create index from all subsequences consisting of k characters
            spaced ival positions apart. """
        self.k = k
        self.ival = ival
        self.index = []
        self.span = 1 + ival * (k - 1)
        for i in range(len(t) - self.span + 1):
            self.index.append((t[i:i+self.span:ival], i))
        self.index.sort()
        
    def query(self, p):
        """ Return index hits for first subseq of p """
        subseq = p[:self.span:self.ival]
        i = bisect.bisect_left(self.index, (subseq, -1))
        hits = []
        while i < len(self.index):
            if self.index[i][0] != subseq:
                break
            hits.append(self.index[i][1])
            i += 1
        return hits

def read_fasta(filename):
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('>'):
                continue
            sequence.append(line.strip())
    return ''.join(sequence)

def query_subseq_index(p, t, subseq_index, max_mismatches=2):
    occurrences = set()
    total_hits = 0
    
    # ival = 3 means 3 unique staggered subsequences starting at indices 0, 1, 2
    for start_offset in range(3):
        # We pass the pattern sliced from the offset onward
        hits = subseq_index.query(p[start_offset:])
        total_hits += len(hits)
        
        for hit in hits:
            start_t = hit - start_offset
            if start_t < 0 or start_t + len(p) > len(t):
                continue
                
            mismatches = 0
            for j in range(len(p)):
                if t[start_t + j] != p[j]:
                    mismatches += 1
                    if mismatches > max_mismatches:
                        break
            if mismatches <= max_mismatches:
                occurrences.add(start_t)
                
    return sorted(list(occurrences)), total_hits



In [12]:
# --- Execution ---
t = read_fasta('chr1.GRCh38.excerpt.fasta')
subseq_idx = SubseqIndex(t, k=8, ival=3)
p = 'GGCGCGGTGGCTCACGCCTGTAAT'
matches, hits_count = query_subseq_index(p, t, subseq_idx)

print(f"Total Matches Found: {len(matches)}")
print(f"Total Subsequence Index Hits Traversed: {hits_count}")

Total Matches Found: 19
Total Subsequence Index Hits Traversed: 79
